1) Data exploration
Here is the Kaggle link to the dataset : https://www.kaggle.com/datasets/tarique7/daily-crude-price-dataset

In [1]:
# libraries import for this project
import pandas as pd
import numpy as np
import matplotlib. pyplot as plt
import matplotlib. dates as mdates
import seaborn as sns
from scipy. stats import norm
from statsmodels. tsa. stattools import adfuller, kpss
from statsmodels. tsa. seasonal import seasonal_decompose
from statsmodels. graphics. tsaplots import month_plot, plot_acf, plot_pacf
from statsmodels. tsa. holtwinters import SimpleExpSmoothing, Holt, ExponentialSmoothing
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, mean_absolute_error
from statsmodels. tsa. statespace. sarimax import SARIMAX
from statsmodels. tsa. arima. model import ARIMA
from statsmodels. tsa. stattools import arma_order_select_ic
from sklearn. model_selection import TimeSeriesSplit
from sklearn. preprocessing import MinMaxScaler
from itertools import product
from math import sqrt
from darts. timeseries import TimeSeries
from darts. utils. timeseries_generation import datetime_attribute_timeseries
from darts. dataprocessing. transformers import Scaler
from darts. models import RNNModel, NBEATSModel
from pandas.tseries.offsets import Day, BDay
import kagglehub


C:\Users\alexi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The StatsForecast module could not be imported. To enable support for the AutoARIMA, AutoETS and Croston models, please consider installing it.


In [ ]:
# dataset import
path = kagglehub.dataset_download("tarique7/daily-crude-price-dataset")

print("Path to dataset files:", path)



100%|██████████| 83.7k/83.7k [00:00<00:00, 547kB/s]

Extracting files...
Path to dataset files: C:\Users\alexi\.cache\kagglehub\datasets\tarique7\daily-crude-price-dataset\versions\2


In [ ]:
#changing the path to the csv file
path = "./data/Crude_oil_data.csv"

# loading the dataset
df = pd. read_csv(path)

#dataset exploration
print(df.info())


#Found 11 missing values in the "Vol." column, we will take the mean of the column on a 30-day window to fill these missing values
def convert_volume(vol):
    if pd.isna(vol):
        return np.nan
    if isinstance(vol, str):
        vol = vol.upper().replace('K', 'e3').replace('M', 'e6').replace('B', 'e9')
        try:
            return float(vol)
        except ValueError:
            return np.nan
    return float(vol)

df['Vol.'] = df['Vol.'].apply(convert_volume)

#converting the "Date" column to datetime format
df['Date'] = pd. to_datetime(df['Date'], format='%m/%d/%Y')

#Setting the "Date" column as the index of the dataframe
df = df. set_index('Date')
df = df.sort_index()

df['Vol.'] = df['Vol.'].fillna(df['Vol.'].rolling(window=30, min_periods=1).mean())

#dropping the "Change %" column since it is not useful for our analysis
df = df.drop(columns=['Change %'])

print(df.info())






<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Date      5000 non-null   object 
 1   Price     5000 non-null   float64
 2   Open      5000 non-null   float64
 3   High      5000 non-null   float64
 4   Low       5000 non-null   float64
 5   Vol.      4989 non-null   object 
 6   Change %  5000 non-null   object 
dtypes: float64(4), object(3)
memory usage: 273.6+ KB
None
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 5000 entries, 1987-01-08 to 2006-12-13
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Price   5000 non-null   float64
 1   Open    5000 non-null   float64
 2   High    5000 non-null   float64
 3   Low     5000 non-null   float64
 4   Vol.    5000 non-null   float64
dtypes: float64(5)
memory usage: 234.4 KB
None


The data are now clean, filling the na found in the volume column with a 30-days window volume mean.  